# btp-baseline+: HazeGen / DiffDehaze CUDA low-memory inference
Run cells **one at a time**, in order, with a GPU and Internet enabled. Attach the private source dataset and private weight dataset first. This notebook has **not been run on Kaggle**. No paper scores are claimed.

Reference runs execute the original upstream scripts at 50 steps, float32, batch size 1. The authors' short-edge resize, zero padding and output resize are retained. CPU/MPS preview settings and AlignOp patches from the local viewer are not used. Torch 2.6 + SDPA are compatibility choices rather than an exact recreation of the authors' Torch 2.2.2/xformers runtime.

Low-memory mode retains float32, original geometry and 50 steps, but freezes weights and offloads idle modules. It reduces GPU residency, not parameter count. GPU fit remains experimental until this run succeeds. Previews are disabled initially.


In [ ]:
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from IPython.display import FileLink, Image, clear_output, display

subprocess.run(["nvidia-smi"], check=True)
# Attached private weights dataset; update the version if the dataset changes.
WEIGHTS_DATASET_VERSION = "cubingultimate/btp-paper-weights-2/versions/1"
assert not WEIGHTS_DATASET_VERSION.startswith("EDIT_ME"), (
    "Set the dataset version above first"
)
ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working/btp-results")
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Restore and verify the uploaded source bundle
Kaggle may keep the ZIP or expand it; both layouts are supported. Existing working copies are checked rather than silently overwritten.

In [ ]:
archives = list(Path("/kaggle/input").rglob("btp-source.bundle"))
if not archives:
    archives = list(Path("/kaggle/input").rglob("btp-kaggle.zip"))
if ROOT.exists():
    shutil.move(str(ROOT), str(ROOT.with_name("BTP-backup-" + str(time.time_ns()))))
if not ROOT.exists():
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.namelist():
                destination = (Path("/kaggle/working") / item).resolve()
                if not destination.is_relative_to(Path("/kaggle/working")):
                    raise ValueError("Unsafe archive member")
            archive.extractall("/kaggle/working")
    else:
        manifests = list(Path("/kaggle/input").rglob("source-manifest.json"))
        assert len(manifests) == 1, "Attach exactly one BTP source dataset"
        shutil.copytree(manifests[0].parent, ROOT)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
# Recover the exact upstream gzip if a dataset upload expanded it to .txt.
vocab_relative = "upstream/Learning-Hazing-to-Dehazing/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
vocab = ROOT / vocab_relative
if not vocab.exists() and vocab.with_suffix("").exists():
    import gzip
    import urllib.request

    commit = snapshot["upstream_commit"]
    assert len(commit) == 40 and all(c in "0123456789abcdef" for c in commit)
    url = (
        "https://raw.githubusercontent.com/ruiyi-w/Learning-Hazing-to-Dehazing/"
        + commit
        + "/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
    )
    with urllib.request.urlopen(url, timeout=60) as response:
        original_gzip = response.read()
    assert (
        hashlib.sha256(original_gzip).hexdigest() == snapshot["files"][vocab_relative]
    ), "Tokenizer download hash mismatch"
    assert gzip.decompress(original_gzip) == vocab.with_suffix("").read_bytes(), (
        "Expanded tokenizer differs from upstream"
    )
    vocab.write_bytes(original_gzip)
    print("Recovered verified tokenizer gzip:", vocab)

for relative, expected in snapshot["files"].items():
    # Finder metadata is not source and can be omitted during dataset upload.
    if Path(relative).name == ".DS_Store":
        continue
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, (
        relative
    )
print("Verified source snapshot:", snapshot["upstream_commit"])

## 2. Create an isolated Python 3.10 CUDA runtime
This downloads Python, CUDA-enabled PyTorch and dependencies. The environment is in `/tmp` so it is not included in saved notebook outputs. It leaves Kaggle's notebook-kernel packages intact.

In [ ]:
uv_binary = shutil.which("uv")
if uv_binary is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    UV = [sys.executable, "-m", "uv"]
else:
    UV = [uv_binary]
ENV = Path("/tmp/btp-kaggle-env")
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run(UV + ["venv", "--python", "3.10", str(ENV)], check=True)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = os.environ.copy()
ENVIRONMENT.update(
    PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0", ATTN_MODE="sdp"
)
subprocess.run(
    RUN
    + [
        "-c",
        "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory/1e9)",
    ],
    check=True,
    env=ENVIRONMENT,
)
with (OUTPUT / "runtime-packages.txt").open("w") as file:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], check=True, stdout=file
    )

# Tiny CPU check in the existing inference environment; no extra packages.
subprocess.run(
    RUN + [str(ROOT / "scripts/smoke_memory.py")], check=True, env=ENVIRONMENT, cwd=ROOT
)

## 3. Locate and hash the three checkpoints
Weights remain on the read-only input mount. The two supplied Baidu files are byte-identical to our previous copies; this step detects incomplete/wrong uploads.

In [ ]:
catalog = json.loads((ROOT / "configs/checkpoints.json").read_text())
paths = {}
for name, spec in catalog.items():
    candidates = list(Path("/kaggle/input").rglob(name))
    assert len(candidates) == 1, f"Expected exactly one {name}, found {len(candidates)}"
    path = candidates[0]
    assert path.stat().st_size == spec["bytes"], f"Incomplete upload: {name}"
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == spec["sha256"], f"Hash mismatch: {name}"
    paths[name] = path
    print("Verified", name, digest.hexdigest())
# Files can be in different dataset directories; symlink them into a common folder.
WEIGHTS = Path("/tmp/btp-reference-weights")
WEIGHTS.mkdir(exist_ok=True)
for name, path in paths.items():
    target = WEIGHTS / name
    if target.is_symlink():
        assert target.resolve() == path.resolve(), (
            "Restart session after changing datasets"
        )
    elif target.exists():
        raise FileExistsError(target)
    else:
        target.symlink_to(path)
(OUTPUT / "weights-verification.json").write_text(
    json.dumps(
        {
            "dataset_version": WEIGHTS_DATASET_VERSION,
            "files": {
                name: {"path": str(path), **catalog[name]}
                for name, path in paths.items()
            },
        },
        indent=2,
    )
)

## 4. Tiny CPU smoke first
Random tiny networks test plumbing only. Their images are not pretrained results. This cell does not run training or a large model.

In [ ]:
subprocess.run(RUN + ["-m", "btp.model_smoke"], cwd=ROOT, env=ENVIRONMENT, check=True)

## 5. Define live single-image execution
Each case runs in a fresh process, releasing the previous model's GPU memory. Diagnostic previews decode noisy sampler states and are **not final images**. Interrupted runs are terminated. Use a new output name to rerun; existing experiments are never overwritten.

In [ ]:
def available_case_name(name):
    """Preserve earlier results and logs when a notebook cell is rerun."""
    candidate = name
    attempt = 0
    while (OUTPUT / candidate).exists() or (OUTPUT / f"{candidate}.log").exists():
        attempt += 1
        candidate = f"{name}-rerun-{attempt:03d}"
    return candidate


def run_case(name, mode, image, preview_every=0):
    name = available_case_name(name)
    output = OUTPUT / name
    print("Saving new case:", output)
    command = RUN + [
        "-u",
        "-m",
        "btp.reference",
        "--low-memory",
        "--mode",
        mode,
        "--input",
        str(image),
        "--weights",
        str(WEIGHTS),
        "--output",
        str(output),
        "--seed",
        "42",
        "--dataset-version",
        WEIGHTS_DATASET_VERSION,
        "--preview-every",
        str(preview_every),
    ]
    log_path = OUTPUT / f"{name}.log"
    process = None
    try:
        with log_path.open("w") as log:
            process = subprocess.Popen(
                command,
                cwd=ROOT,
                env=ENVIRONMENT,
                stdout=log,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            while process.poll() is None:
                clear_output(wait=True)
                status = output / "status.json"
                print(
                    name,
                    json.loads(status.read_text())
                    if status.exists()
                    else "Starting runtime…",
                )
                print("\n".join(log_path.read_text(errors="replace").splitlines()[-6:]))
                frames = sorted(output.glob("latent-*.png")) if output.exists() else []
                if frames:
                    print("Diagnostic noisy-latent decode:")
                    display(Image(filename=str(frames[-1]), width=600))
                time.sleep(2)
            if process.returncode:
                raise RuntimeError(log_path.read_text(errors="replace")[-12000:])
    finally:
        if process is not None and process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait()
    clear_output(wait=True)
    print("Completed:", output)
    print("Input:")
    display(Image(filename=str(output / "input.png"), width=600))
    print("Final model output:")
    display(Image(filename=str(output / "final/input.png"), width=600))
    return output / "final/input.png"

## 6. HazeGen — clean photograph → hazy photograph
The bundled garden photo is a functional example, not a paper benchmark. You can replace CLEAN with your own clean image mounted under `/kaggle/input`. HazeGen uses the authors' prompt and `w=0.85`, with 50 steps.

In [ ]:
CLEAN = ROOT / "examples/clean.png"
GENERATED_HAZE = run_case("01-hazegen", "hazegen", CLEAN)

## 7. DiffDehaze — upstream real hazy example, standard sampler
Start with one supplied example. This is qualitative inference; the example has no paired clean ground truth here.

In [ ]:
HAZY = ROOT / "upstream/Learning-Hazing-to-Dehazing/inputs/1.png"
SPACED = run_case("02-spaced", "spaced", HAZY)

## 8. DiffDehaze — same real image, AccSamp
Uses the original `tau=0.8`, `omega=0.6`, fidelity scale `s=0.1`, and the 50-step schedule. AccSamp makes fewer actual denoiser calls than a full 50-call spaced trajectory. Its original AlignOp behavior is preserved.

In [ ]:
ACCSAMP = run_case("03-accsamp", "accsamp", HAZY)

## 9. Complete pipeline — generated haze → DiffDehaze
Optional after inspecting the previous results. An end-to-end roundtrip is not a replacement for the paper's real-world dataset evaluation.

In [ ]:
ROUNDTRIP = run_case("04-roundtrip", "accsamp", GENERATED_HAZE)

## 10. Save outputs before ending the session
Save a Kaggle notebook version with outputs and download this ZIP. It contains actual saved images, logs, manifests, hashes and runtime packages—not model weights or the virtual environment. GPU outputs have not been pre-filled into this notebook.

In [ ]:
archive = shutil.make_archive("/kaggle/working/btp-results", "zip", OUTPUT)
display(FileLink(archive))
print(
    "Use Kaggle's Output/Files panel if the link is unavailable. Save a notebook version before ending the session."
)